# SDXL Convert Phase 1 (GPU, fp16)

Runs `prepare_data_sdxl.py` on Kaggle T4x2 GPU. CUDA auto-enables fp16 → 5-10x faster than CPU fp32.

Output (saved to `/kaggle/working/`, becomes kernel output dataset):
- `data_sdxl.pkl`
- `images_sdxl/*.png`
- `model.safetensors` (forwarded to Phase 2 kernel to avoid re-download)

In [ ]:
# Config — placeholders patched by GH Actions before push
CIVITAI_VERSION_ID = '__CIVITAI_VERSION_ID__'
MODEL_NAME         = '__MODEL_NAME__'
REALISTIC          = '__REALISTIC__' == 'true'
print(f'civitai={CIVITAI_VERSION_ID} name={MODEL_NAME} realistic={REALISTIC}')

In [ ]:
import os, subprocess
!nvidia-smi
!free -h
!df -h /kaggle/working
import torch
print(f'CUDA available: {torch.cuda.is_available()}')

In [ ]:
# Civitai API key — injected by GH Actions from repo secret CIVITAI_API_KEY
import os
CIVITAI_API_KEY = '__CIVITAI_API_KEY__'
os.environ['CIVITAI_API_KEY'] = CIVITAI_API_KEY
os.environ['CIVITAI_VERSION_ID'] = CIVITAI_VERSION_ID
os.environ['MODEL_PATH'] = '/kaggle/working/model.safetensors'
print('API key injected')

In [ ]:
# Install tools + convertsdxl
!apt-get install -y unzip zip curl time > /dev/null
!curl -LsSf https://astral.sh/uv/install.sh | sh > /dev/null 2>&1
import os
os.environ['PATH'] = f"/root/.local/bin:{os.environ['PATH']}"

!rm -rf /tmp/convertsdxl.zip /tmp/convertsdxl_unzip
!curl -sL -o /tmp/convertsdxl.zip 'https://chino.icu/local-dream/convertsdxl.zip'
!mkdir -p /tmp/convertsdxl_unzip
!unzip -q /tmp/convertsdxl.zip -d /tmp/convertsdxl_unzip

import subprocess
root = subprocess.check_output("find /tmp/convertsdxl_unzip -maxdepth 3 -name 'export_sdxl.sh' -printf '%h\\n' | head -1", shell=True).decode().strip()
assert root, 'export_sdxl.sh not found'
NPUCONVERT_DIR = os.path.abspath(root)
os.environ['NPUCONVERT_DIR'] = NPUCONVERT_DIR
print(f'NPUCONVERT_DIR: {NPUCONVERT_DIR}')

In [ ]:
# Download Civitai model to /kaggle/working so it propagates to Phase 2 kernel
import os
if not os.path.exists(os.environ['MODEL_PATH']) or os.path.getsize(os.environ['MODEL_PATH']) < int(1e9):
    !curl -L -H "Authorization: Bearer $CIVITAI_API_KEY" \
        -o "$MODEL_PATH" --fail --retry 5 --retry-delay 5 \
        "https://civitai.com/api/download/models/$CIVITAI_VERSION_ID"
!ls -lh $MODEL_PATH
size = os.path.getsize(os.environ['MODEL_PATH'])
assert size > int(1e9), f'Downloaded file is only {size} bytes (404 or wrong key?)'

In [ ]:
# Use uv venv (as the original export_sdxl.sh does) for isolated env with pinned deps.
# After uv sync installs torch==2.5.1+cpu, force-reinstall the same version CUDA build from
# cu121 index. torch 2.5.1 supports sm_50..sm_90 → works on Kaggle P100 (sm_60) AND T4 (sm_75).
import os
os.chdir(os.environ['NPUCONVERT_DIR'])
!uv venv -p 3.10.17 --clear
!. .venv/bin/activate && uv sync
!. .venv/bin/activate && pip install --force-reinstall --no-deps torch==2.5.1 --index-url https://download.pytorch.org/whl/cu121

In [ ]:
# CUDA smoke test — fail fast if torch/GPU mismatch (P100 sm_60 vs new torch sm_70+)
!. .venv/bin/activate && python - <<'PY'
import torch, sys
print(f'torch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if not torch.cuda.is_available():
    sys.exit('CUDA not available — abort to avoid slow CPU fallback')
print(f'device: {torch.cuda.get_device_name(0)}  capability: sm_{torch.cuda.get_device_capability(0)[0]}{torch.cuda.get_device_capability(0)[1]}')
try:
    t = torch.zeros(1, device='cuda'); (t + 1).cpu()
    print('CUDA kernel launch OK')
except Exception as e:
    sys.exit(f'CUDA kernel launch failed: {e}')
PY

In [ ]:
# Phase 1 — prepare_data_sdxl.py in uv venv (torch 2.5.1+cu121)
import time, os
t0 = time.time()
REALISTIC_FLAG = '--realistic' if REALISTIC else ''
os.environ['REALISTIC_FLAG'] = REALISTIC_FLAG
os.chdir(os.environ['NPUCONVERT_DIR'])
!. .venv/bin/activate && /usr/bin/time -v python prepare_data_sdxl.py \
    --model_path "$MODEL_PATH" $REALISTIC_FLAG \
    2>&1 | tee /kaggle/working/phase1.log
print(f'Phase 1 elapsed: {int(time.time()-t0)}s')

In [ ]:
# Copy outputs to /kaggle/working so they become kernel output for Phase 2
import os, shutil
os.makedirs('/kaggle/working/phase1_out', exist_ok=True)
shutil.copy(f'{os.environ["NPUCONVERT_DIR"]}/data_sdxl.pkl', '/kaggle/working/phase1_out/data_sdxl.pkl')
if os.path.exists(f'{os.environ["NPUCONVERT_DIR"]}/images_sdxl'):
    if os.path.exists('/kaggle/working/phase1_out/images_sdxl'):
        shutil.rmtree('/kaggle/working/phase1_out/images_sdxl')
    shutil.copytree(f'{os.environ["NPUCONVERT_DIR"]}/images_sdxl', '/kaggle/working/phase1_out/images_sdxl')
!ls -lh /kaggle/working/
!du -sh /kaggle/working/phase1_out